# Lab 5: Tool-Using Agent (Calculator, Weather, Time)

**AI Agent Lab 5 of 8**

---

## What this lab does

In this lab you will build an agent that reads a user's request in plain English, decides which tool to use, calls that tool, and returns the answer. The agent has three tools: a calculator, a weather lookup (with hardcoded data for five cities), and a clock. The key idea is that tools are just functions stored in a dictionary, and the agent decides which function to call based on keyword matching. You will see the agent's reasoning printed step by step.

## What you will learn

**Python concepts:**
- How to store functions in a dictionary and call them dynamically
- How to use `eval()` safely for simple maths expressions
- How to parse a number from a string using `re.findall()`
- How to build a dispatcher using a dictionary of tool functions
- How to handle requests the agent cannot answer (fallback)

**Agent concepts:**
- Tool selection based on intent detection
- The ReAct pattern: Reason, then Act
- Graceful degradation with fallback responses
- How real AI agents (like GPT-4 with tools) decide which function to call

## Important things to know before starting

- **What is a tool in agent terms?** A tool is a function the agent can call to get information or perform an action. The agent must decide which tool fits the request.
- **What is a dispatcher?** A dispatcher is a dictionary that maps tool names to tool functions. Instead of a long if-elif chain, the agent looks up the right function by name.
- **Safety note on eval():** We only use `eval()` on strings that contain only digits, operators, and spaces - never on user input that has not been checked. This is important to understand.
- **This lab runs completely offline.** Weather data is hardcoded, not fetched from the internet.

## How the agent works

```
User request (plain English string)
        |
        v
  Reason: which tool does this request need?
  (keyword matching against tool descriptions)
        |
        v
  Act: look up the tool function in the dispatcher dictionary
  and call it with the extracted arguments
        |
        v
  Report: print the tool result with clear labels
```

---
## Step 0: Libraries and Setup

This lab uses **only built-in Python libraries**.

We use:
- `re` - to extract numbers and detect keywords in user requests
- `datetime` - to get the current time for the clock tool
- `sys` - to print the Python version

In [ ]:
# No installation needed
# Optional: # !pip install nothing

import sys
print('Setup check passed!')
print(f'Python version: {sys.version}')

---
## Importing Libraries

| Library | What it is used for in this lab |
|---|---|
| `re` | Extracts numbers from text and detects tool keywords |
| `datetime` | Used by the `get_time()` tool to return the current time |

In [ ]:
import re        # used to extract numbers and detect tool type in user requests
import datetime # used by the get_time tool

print('Libraries imported successfully.')

---
## Step 1: Build the Tool Functions

**What this block does:** We define the three tool functions the agent can use:
- `calculate(expression_string)` - evaluates a maths expression safely
- `get_weather(city_name)` - returns hardcoded weather for one of five cities
- `get_time()` - returns the current time from the system clock

**Why we need it:** These are the agent's capabilities. Without tools, the agent can only repeat what it knows; with tools, it can compute, look up, and retrieve information.

**What to expect as output:** Each tool is defined and a short self-test is printed.

In [ ]:
# --- TOOL 1: Calculator ---
# Evaluates a simple maths expression given as a string.
# Safety: we only allow digits, spaces, and basic operators.

def calculate(expression_string):
    """
    Evaluates a maths expression and returns the result.

    Takes in:
      expression_string - a maths expression like '15 * 3 + 7' (string)

    Returns:
      The numeric result as a string, or an error message.
    """
    # Only allow safe characters to prevent code injection
    safe_characters_pattern = r'^[0-9\s\+\-\*\/\.\(\)]+$'
    if not re.match(safe_characters_pattern, expression_string):
        return 'Error: expression contains unsupported characters.'

    try:
        calculation_result = eval(expression_string)
        return f'{expression_string} = {calculation_result}'
    except Exception:
        return f'Error: could not evaluate "{expression_string}"'


print(f'Calculator test: {calculate("15 * 3 + 7")}')

In [ ]:
# --- TOOL 2: Weather Lookup ---
# Returns hardcoded weather for five cities.
# SIMULATED SERVICE: In a real system this would call a weather API.

# Hardcoded weather data - stored as a dictionary of city -> weather info
hardcoded_weather_data = {
    'london': {
        'condition': 'overcast with light rain',
        'temperature_celsius': 13,
        'humidity_percent': 82
    },
    'dubai': {
        'condition': 'sunny and very hot',
        'temperature_celsius': 39,
        'humidity_percent': 55
    },
    'tokyo': {
        'condition': 'partly cloudy',
        'temperature_celsius': 22,
        'humidity_percent': 68
    },
    'new york': {
        'condition': 'clear and sunny',
        'temperature_celsius': 18,
        'humidity_percent': 45
    },
    'karachi': {
        'condition': 'hot and humid',
        'temperature_celsius': 34,
        'humidity_percent': 78
    }
}


def get_weather(city_name):
    """
    Returns hardcoded weather information for a given city.

    Takes in:
      city_name - the name of the city (string, case-insensitive)

    Returns:
      A weather summary string, or an error if the city is not in the database.
    """
    lowered_city_name = city_name.lower().strip()
    if lowered_city_name not in hardcoded_weather_data:
        known_cities = list(hardcoded_weather_data.keys())
        return f'Weather data not available for "{city_name}". Known cities: {known_cities}'

    weather_info = hardcoded_weather_data[lowered_city_name]
    return (
        f'Weather in {city_name.title()}: {weather_info["condition"]}, '
        f'{weather_info["temperature_celsius"]}C, '
        f'humidity {weather_info["humidity_percent"]}%'
    )


print(f'Weather test: {get_weather("Tokyo")}')

In [ ]:
# --- TOOL 3: Clock ---
# Returns the current local time.

def get_time():
    """
    Returns the current local time as a formatted string.

    Takes in: nothing
    Returns: current time string in 'HH:MM:SS' format
    """
    current_time = datetime.datetime.now()
    return f'Current local time is: {current_time.strftime("%I:%M %p on %A, %B %d, %Y")}'


print(f'Time test: {get_time()}')

---
## Step 2: Build the Tool Dispatcher Dictionary

**What this block does:** We create a dictionary called `tool_dispatcher` that maps tool names (strings) to the actual tool functions. This is the agent's toolkit.

**Why we need it:** Using a dictionary instead of a long if-elif chain makes the code cleaner and easier to extend. Adding a new tool is just one line: a new key-value pair.

**What to expect as output:** The list of registered tools is printed.

In [ ]:
# Map tool names to their functions.
# To add a new tool: define the function above, then add it here.
tool_dispatcher = {
    'calculate': calculate,   # maths tool - evaluates an expression
    'get_weather': get_weather, # weather tool - returns weather for a city
    'get_time': get_time       # clock tool - returns current time
}

print(f'Tool dispatcher ready. Registered tools: {list(tool_dispatcher.keys())}')

---
## Step 3: Build the Tool Selector

**What this block does:** We define `select_tool_for_request()`. It reads a user request and returns the name of the best tool to use. It also extracts the argument to pass to the tool (a city name or a maths expression).

**Why we need it:** This is the 'reasoning' step of the agent. Before acting, the agent must decide WHAT to do. Printing its reasoning makes the decision transparent.

**What to expect as output:** The function is defined and a quick test is printed.

In [ ]:
def select_tool_for_request(user_request_text):
    """
    Reads a user request and decides which tool to use.

    Returns a tuple: (tool_name, tool_argument)
    - tool_name is a string matching a key in tool_dispatcher, or 'none'
    - tool_argument is the string to pass to the tool, or None

    Takes in:
      user_request_text - the user's request in plain English (string)

    Returns:
      A tuple (tool_name_string, tool_argument_or_None)
    """

    lowered_request = user_request_text.lower()

    # --- Check for time/clock requests ---
    if any(word in lowered_request for word in ['time', 'clock', 'hour', "what's the time"]):
        print('Agent thinking: this looks like a time request, using tool get_time')
        return ('get_time', None)

    # --- Check for weather requests ---
    if any(word in lowered_request for word in ['weather', 'temperature', 'forecast', 'humid', 'rain', 'sunny']):
        # Try to extract the city name - look for known city names
        known_city_names = list(hardcoded_weather_data.keys())
        extracted_city = None
        for city_name in known_city_names:
            if city_name in lowered_request:
                extracted_city = city_name
                break
        if extracted_city:
            print(f'Agent thinking: this looks like a weather request, using tool get_weather for {extracted_city}')
            return ('get_weather', extracted_city)
        else:
            print('Agent thinking: weather request detected but no known city found.')
            return ('get_weather', 'unknown')

    # --- Check for calculation requests ---
    calculation_keywords = ['calculate', 'compute', 'what is', 'how much is', '+', '-', '*', '/', 'times', 'plus', 'minus', 'divided']
    if any(word in lowered_request for word in calculation_keywords):
        # Extract the mathematical expression (numbers and operators)
        extracted_numbers = re.findall(r'[\d\.]+', user_request_text)
        operator_map = {'plus': '+', 'minus': '-', 'times': '*', 'divided by': '/', 'multiplied by': '*'}
        math_expression = user_request_text
        for word_operator, symbol_operator in operator_map.items():
            math_expression = math_expression.replace(word_operator, symbol_operator)
        # Extract just the numeric part of the expression
        numeric_expression = re.sub(r'[^0-9\s\+\-\*\/\.\(\)]', '', math_expression).strip()
        if numeric_expression:
            print(f'Agent thinking: this looks like a calculation request, using tool calculate with "{numeric_expression}"')
            return ('calculate', numeric_expression)

    # --- No tool matched ---
    print('Agent thinking: I do not recognise this request type. Using fallback.')
    return ('none', None)


# Quick test
test_tool, test_arg = select_tool_for_request('What is the weather in London?')
print(f'Tool selected: {test_tool}, argument: {test_arg}')

---
## Step 4: Build the Tool-Using Agent

**What this block does:** We define `tool_using_agent()`, the main agent function. It calls `select_tool_for_request()` to decide what to do, then looks up the tool in `tool_dispatcher`, calls it, and prints the result.

**Why we need it:** This ties together reasoning and action. The agent does not hardcode what to do - it looks it up dynamically.

**What to expect as output:** The function is defined.

In [ ]:
def tool_using_agent(user_request_text):
    """
    The main Tool-Using Agent.

    Reads a user request, selects the right tool, calls it,
    and prints the result.

    Takes in:
      user_request_text - the user's plain-English request (string)

    Returns:
      Nothing - prints the result to the screen.
    """

    print('\n' + '=' * 60)
    print(f'USER REQUEST: {user_request_text}')
    print('-' * 60)

    # Step 1: Reason - select the right tool
    selected_tool_name, tool_argument = select_tool_for_request(user_request_text)

    # Step 2: Act - call the tool
    if selected_tool_name == 'none':
        # Fallback: the agent could not identify a tool for this request
        tool_result = (
            'I am not sure how to help with that. '
            'I can help with: calculations, weather for London/Dubai/Tokyo/New York/Karachi, or the current time.'
        )
    elif selected_tool_name == 'get_time':
        # The time tool takes no arguments
        tool_function = tool_dispatcher['get_time']
        tool_result = tool_function()
    else:
        # All other tools take one argument
        tool_function = tool_dispatcher[selected_tool_name]
        tool_result = tool_function(tool_argument)

    # Step 3: Report - print the answer
    print(f'AGENT ANSWER: {tool_result}')
    print('=' * 60)


print('tool_using_agent() defined successfully.')

---
## Step 5: Run the Agent on Sample Requests

**What this block does:** We define a list of eight sample user requests - covering all three tools and some edge cases - and run the agent on each one.

**Why we need it:** This is the main demo. You will see the agent reason about each request, select the right tool, call it, and report the answer.

**What to expect as output:** Eight agent runs, each showing the reasoning step and the result.

In [ ]:
# These sample requests exercise all three tools and the fallback case
list_of_sample_user_requests = [
    'What is the weather like in London today?',
    'Calculate 250 * 4 + 150',
    'What time is it right now?',
    'Tell me the temperature in Karachi',
    'How much is 1024 / 8?',
    'Is it raining in Tokyo?',
    'What is 99 plus 1?',
    'Can you book me a flight to Paris?'   # fallback - agent cannot handle this
]

# Run the agent on every request
for single_request in list_of_sample_user_requests:
    tool_using_agent(single_request)

---
## Step 6: Edge Case - Unknown City

**What this block does:** We test what happens when the user asks about a city that is not in the hardcoded weather database. The agent should still try to use the weather tool and return a clear error from the tool.

**Why we need it:** Agents should fail gracefully even inside a tool call, not just at the tool selection stage.

**What to expect as output:** The agent's thinking and the weather tool's error message.

In [ ]:
# Edge case: weather request for a city not in the database
print('EDGE CASE TEST: Unknown city')
tool_using_agent('What is the weather in Sydney?')

---
## Summary

You built a **Tool-Using Agent** that reads plain-English requests, selects the right tool, and calls it dynamically using a dispatcher dictionary.

### Key takeaways
- **Functions in dictionaries:** Storing functions as dictionary values lets you look up and call them by name - this is a fundamental agent design pattern.
- **Reasoning before acting:** The agent prints its thinking at every step, making its decisions transparent and debuggable.
- **Tool isolation:** Each tool function is self-contained and knows nothing about the agent - this makes tools easy to test and replace independently.
- **Fallback is mandatory:** When no tool matches, the agent must say so clearly rather than crashing or returning wrong data.
- **This mirrors real function-calling AI:** Agents like GPT-4 with function calling work exactly this way: they pick a function, extract arguments, call it, and report the result.

### Connection to the next lab

In **Lab 6**, you will build a Daily Planner Agent. Instead of answering questions, it will manage a to-do list - adding tasks, marking them done, removing them, and offering AI-style prioritisation suggestions.

---

## Try It Yourself

1. **Add a fourth tool:** Write a function called `convert_currency(amount_and_currency)` that converts between two hardcoded currencies (e.g., USD to EUR at a fixed rate). Add it to `tool_dispatcher` and update `select_tool_for_request()` to detect currency requests.

2. **Add more cities:** Add three more cities to `hardcoded_weather_data` and test the agent with requests mentioning those cities.

3. **Improve the calculator:** Currently the agent strips everything except digits and operators to extract the expression. Try making it smarter: look for the pattern 'what is X operation Y' and extract X, the operator word, and Y separately.